# Data Lakehouse Architecture — Notes

Comprehensive overview: evolution from Data Warehouse → Data Lake → Lakehouse, plus Delta Lake internals and real-world scenario guide.

## 1. Data Warehouse

**Concept:** Emerged in the 1980s as a centralized repository for **structured and semi-structured data**.

**Key Function:** Designed for operational data reporting and business intelligence (BI).

**Limitations:**
- Cannot handle **unstructured data** (images, videos, raw logs) — ~90% of modern data.
- High storage and maintenance costs.
- Slow data ingestion due to rigid **ETL** (Extract, Transform, Load) requirements.
- Lacks support for modern **Data Science** and **Machine Learning (ML)** workloads.

<img src="/Volumes/sample/demo/images/data-warehouse.png" with="auto">

## 2. Data Lake 

**Concept:** Introduced around 2010 to solve the rigidity of warehouses.

**Key Function:** A low-cost, scalable repository for **all data types** (structured, semi-structured, unstructured).

**Advantages:** Uses open-source file formats (*Parquet*, *ORC*, *Avro*) and cost-effective cloud storage.

**Limitations:**
- **Lack of ACID Transactions** → data corruption, partial loads, inconsistent reads.
- **Poor BI Performance** — not optimized for real-time reporting vs. warehouses.
- **Complex Management** — no built-in version control/governance, hard to correct data.

<img src="/Volumes/sample/demo/images/data-lake.png" width="auto">

## 3. Data Lakehouse 

**Concept:** Unified architecture combining the **flexibility & cost-efficiency of a Data Lake** with the **governance & ACID reliability of a Data Warehouse**.

**Foundational Technology:** Built on **Delta Lake**, which adds:
- **ACID Transactions** — reliable read/write operations.
- **Versioning / Time Travel** — revert to prior data versions (auditing, debugging).
- **High Performance** — optimized for BI dashboards *and* AI/ML compute.

**Why it matters:** Eliminates maintaining two separate systems, reducing infra overhead and data duplication, enabling end-to-end processing in one place.

<img src="/Volumes/sample/demo/images/data-lakehouse.png" width="auto">

## 4. Delta Lake — The Engine Behind the Lakehouse

Delta Lake is an **open-source storage layer** sitting on top of your existing data lake (S3, ADLS, GCS) — adds warehouse-like reliability without moving data.

**Core features:**
- **ACID Transactions** — via transaction log (`_delta_log`); no corruption from concurrent reads/writes.
- **Time Travel** — query prior versions with `VERSION AS OF` / `TIMESTAMP AS OF`; useful for audits, rollback, debugging.
- **Schema Enforcement & Evolution** — rejects mismatched writes, allows controlled schema changes (e.g. adding a column).
- **Unified Batch + Streaming** — same table serves as a Structured Streaming sink and a batch query source.
- **Upserts/Merges** — `MERGE INTO` enables CDC, SCD Type 2, deduplication.
- **Optimizations** — `OPTIMIZE` (file compaction), `Z-ORDER` (data clustering) speed up queries; `VACUUM` removes stale files.

> Delta Lake turns "just files in a lake" into queryable, governable **tables** — the technical foundation Unity Catalog and Lakehouse governance build on.

In [0]:
-- Example Delta Lake SQL snippets (run in a Databricks SQL/Python cell)

-- Time travel by version
-- SELECT * FROM sales_delta VERSION AS OF 12;

-- Time travel by timestamp
-- SELECT * FROM sales_delta TIMESTAMP AS OF '2026-07-01';

-- Merge (upsert) for CDC
-- MERGE INTO target t USING updates u ON t.id = u.id
-- WHEN MATCHED THEN UPDATE SET *
-- WHEN NOT MATCHED THEN INSERT *;

-- Optimize + Z-Order
-- OPTIMIZE sales_delta ZORDER BY (customer_id);

-- Vacuum old files
-- VACUUM sales_delta RETAIN 168 HOURS;

## 5. Real-World Scenario Guide

| Scenario | Best Fit | Why |
|---|---|---|
| Finance team needs daily sales dashboards, fixed schema | **Data Warehouse** | Fast SQL, strong consistency, mature BI tool support |
| Storing raw IoT sensor logs, video, clickstream at low cost | **Data Lake** | Cheap storage, schema-on-read, handles any format |
| Company wants ML models AND BI reports off the same customer data, avoiding duplicate pipelines | **Lakehouse (Delta Lake)** | One copy of data serves both SQL analysts and data scientists |
| Need to audit "what did this table look like last Tuesday" for compliance | **Lakehouse (Delta Lake Time Travel)** | Native versioning — warehouses can't do this cheaply, lakes can't do it at all |
| Streaming data (e.g. orders) must be immediately queryable and also feed ML feature pipelines | **Lakehouse** | Delta unifies streaming + batch on the same table |
| Small startup, simple reporting, no ML/AI ambitions | **Data Warehouse** | Lakehouse adds complexity you don't need yet |

**Bottom line:** Lakehouse wins when you need **one source of truth** for both BI and AI/ML without duplicating data across a warehouse and a lake — exactly Databricks' pitch.